### Preprocessing

#### Mattergen Density (Mother Dataset)
- Dataset: Source [Mattergen Alex-MP-20](https://github.com/microsoft/mattergen/tree/main/data-release/alex-mp)
  - Columns: Database (manual) 
  - Reduced Formula (Source)
  - CIF (pmg - Cifwriter with symprec 0.1)
  - energy_above_hull (Source), 
  - Density (g/cm^3) (pmg - structure.density)
- Deduplicated
- Cleaned for CIF augmentation, normalisations:
  - $E_{hull}$ and Density - min-max
- Note: I didnt filter to context length here because it was not implemented yet, but filter to context was flagged as True during model training which effectively does the same thing (less efficient)
- Pushed to huggingface with 90:10 train/eval as c-bone/mattergen_den_ehull

#### Mattergen Density Subsets
- Randomly sampled 1K, 10K, and 100K from the mother dataset
  - df.sample(n=target_size, random_state=1)
- Pushed to Huggingface with 90:10 train/eval as
  - c-bone/mattergen_den_ehull_1K
  - c-bone/mattergen_den_ehull_10K
  - c-bone/mattergen_den_ehull_100K


In [ ]:
import __init__
import matplotlib.pyplot as plt
import pandas as pd

# Density distribution comparison across different dataset sizes
datasets = {
    'Original Data': 'HF-databases/mattergen_dev/mattergen_den_ehull_clean.parquet',
    'Sampled Data 100K': 'HF-databases/mattergen_dev/mattergen_den_ehull_clean_100K.parquet',
    'Sampled Data 10K': 'HF-databases/mattergen_dev/mattergen_den_ehull_clean_10K.parquet',
    'Sampled Data 1K': 'HF-databases/mattergen_dev/mattergen_den_ehull_clean_1K.parquet'
}
data = {name: pd.read_parquet(path) for name, path in datasets.items()}

density_col = 'Density (g/cm^3)'
colors = ['blue', 'orange', 'green', 'red']
labels = list(datasets.keys())

# Plotting histograms
plt.figure(figsize=(10, 6))
for i, (label, df) in enumerate(data.items()):
    plt.hist(df[density_col], bins=30, alpha=0.3, label=label, color=colors[i], density=False)

plt.legend()
plt.yscale('log')
plt.xlabel('Density (g/cm^3)')
plt.ylabel('Frequency')
plt.title('Histogram of Density (g/cm^3)')
plt.show()

#### Training the models

In [ ]:
sizes = ['-1k', '-10k', '-100k', '']
models = ['slider', 'prepend', 'PKV']

In [ ]:
for size in sizes:
    for model in models:
        config_path = f'_config_files/training/conditional/dataset_size_study/mgen_ft-den{size}-{model}.jsonc'
        
        !torchrun --nproc_per_node=2 _train.py --config config_path

#### CIF generation for each

In [ ]:
for size in sizes:
    for model in models:
        # _config_files/generation/conditional/dataset_size_study/mgen_ft-den-1k-PKV_eval.jsonc
        gen_config_path = f'_config_files/generation/conditional/dataset_size_study/mgen_ft-den{size}-{model}_eval.jsonc'

        !python _utils/_generating/generate_CIFs.py --config gen_config_path

### Metrics for each

In [ ]:
for size in sizes:
    for model in models:
        # _artifacts/dataset_size_study/mgen_ft-den-1k-PKV_gen.parquet
        gen_structs_path = f'_artifacts/dataset_size_study/mgen_ft-den{size}-{model}_gen.parquet'

        postprocessed_path = f'_artifacts/dataset_size_study/mgen_ft-den{size}-{model}_post.parquet'

        !python _utils/_metrics/VUN_metrics.py \
            --input_parquet gen_structs_path \
            --huggingface_dataset 'c-bone/mattergen_den_ehull' \
            --load_processed_data 'HF-databases/mattergen_dev/mgen_den_ehull_proc.parquet' \
            --output_parquet postprocessed_path \
            --num_workers 32

        !python _utils/_metrics/property_metrics.py \
            --post_parquet postprocessed_path \
            --output_parquet postprocessed_path \
            --property_targets '["Density (g/cm^3)", "energy_above_hull"]' \
            --num_workers 16 \
            --property1_normaliser "linear" \
            --property2_normaliser "linear" \
            --max_property1 25.494 \
            --min_property1 0.0 \
            --max_property2 0.1 \
            --min_property2 0.0

In [ ]:
for size in sizes:
    for model in models:
        if model == 'PKV' and size == '-1k':
            continue
        
        postprocessed_path = f'_artifacts/dataset_size_study/mgen_ft-den{size}-{model}_post.parquet'
        
        !python _utils/_metrics/mace_ehull.py \
            --post_parquet postprocessed_path \
            --output_parquet postprocessed_path \
            --num_workers 16

In [ ]:
!python _utils/_metrics/mace_ehull.py \
    --post_parquet _artifacts/dataset_size_study/mgen_ft-den-1k-PKV_post.parquet \
    --output_parquet _artifacts/dataset_size_study/mgen_ft-den-1k-PKV_post.parquet \
    --num_workers 16

#### Results & Plots

In [ ]:
import __init__
import pandas as pd
import matplotlib.pyplot as plt
from _utils._notebook_utils.b2_dataset_size_study_utils import plot_density_histogram

base_path = 'HF-databases/mattergen_dev/'
training_dfs = {
    'Original Data': pd.read_parquet(f'{base_path}mattergen_den_ehull_clean.parquet'),
    'Sampled Data 100K': pd.read_parquet(f'{base_path}mattergen_den_ehull_clean_100K.parquet'),
    'Sampled Data 10K': pd.read_parquet(f'{base_path}mattergen_den_ehull_clean_10K.parquet'),
    'Sampled Data 1K': pd.read_parquet(f'{base_path}mattergen_den_ehull_clean_1K.parquet')
}

fig = plot_density_histogram(
    dfs_dict=training_dfs,
    output_path="plots/density/training_density_histogram.png",
    alpha_val=0.6
)

plt.show()

In [ ]:
"""Generates dataset size study metrics and summary plots for density output space and Q_VSUN parity."""

import __init__
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from _utils._notebook_utils import b2_dataset_size_study_utils as b2_utils

importlib.reload(b2_utils)
format_density_metrics_table = b2_utils.format_density_metrics_table
get_metrics_dataset_size_study = b2_utils.get_metrics_dataset_size_study
plot_density_output_space_summary = b2_utils.plot_density_output_space_summary
plot_vsun_parity_grid_density = b2_utils.plot_vsun_parity_grid_density

output_dir = Path("plots/density")
output_dir.mkdir(parents=True, exist_ok=True)

reference_path = Path("HF-databases/mattergen_dev/mattergen_den_ehull_clean.parquet")
sizes = ["-1k", "-10k", "-100k", ""]
models = ["slider", "prepend", "PKV"]

dfs = {}
for size in sizes:
    for model in models:
        postprocessed_path = Path(f"_artifacts/dataset_size_study/mgen_ft-den{size}-{model}_post.parquet")
        dfs[f"mgen_ft-den{size}-{model}"] = pd.read_parquet(postprocessed_path)

train_df = pd.read_parquet(reference_path)

_base = Path("HF-databases/mattergen_dev")
train_dfs = {
    "1k":   pd.read_parquet(_base / "mattergen_den_ehull_clean_1K.parquet"),
    "10k":  pd.read_parquet(_base / "mattergen_den_ehull_clean_10K.parquet"),
    "100k": pd.read_parquet(_base / "mattergen_den_ehull_clean_100K.parquet"),
    "full": train_df,
}


In [ ]:

fig_density_output = plot_density_output_space_summary(
    dfs,
    train_dfs=train_dfs,
    figsize=(25.5, 6.8),
    max_density=25.0,
    bin_width=1.0,
    min_bin_count=10,
    wspace=0.025,
    hspace=0.05,
    axes_linewidth=2.0,
    num_yticks=5,
    num_yticks_ref=3,
    title_fontsize=22,
    label_fontsize=20,
    ticks_fontsize=18,
    legend_fontsize=22,
)
fig_density_output.savefig(output_dir / "dataset_size_study_output_space_counts.png", dpi=300)
plt.show()



In [ ]:

metrics = get_metrics_dataset_size_study(
    dfs,
    train_df=train_df,
)

summary_metrics_df = pd.DataFrame(
    sorted(
        (key, value)
        for key, value in metrics.items()
        if key != "raw_dataframe"
    ),
    columns=["metric", "value"],
)
summary_metrics_df.to_csv(output_dir / "dataset_size_study_summary_metrics.csv", index=False)

In [ ]:

fig_density, metrics_table_density = plot_vsun_parity_grid_density(
    dfs,
    train_dfs=train_dfs,
    figsize=(22, 16),
    line_width=2.5,
    scatter_size=150,
    axes_linewidth=2.0,
    num_yticks=5,
    num_yticks_ref=3,
    title_fontsize=20,
    label_fontsize=18,
    ticks_fontsize=16,
)

formatted_density_table = format_density_metrics_table(metrics_table_density)
formatted_density_table.to_csv(output_dir / "dataset_size_study_parity_table.csv", index=False)
formatted_density_table.to_latex(
    output_dir / "dataset_size_study_parity_table.tex",
    index=False,
    escape=False,
    float_format="%.2f",
)
fig_density.savefig(output_dir / "dataset_size_study_main.png", dpi=300)
plt.show()

print()
print(formatted_density_table.to_string(index=False))
